In [9]:
%%writefile main.cpp
#include <iostream>
#include <cmath>
#include <fstream>
#include <sstream>
#include <string>
using namespace std;



double func_cubic_spline(double x_value, double x[], double y[], int n_points) {
    const int MAX_NODES = 100;

    if (n_points < 3) {
        throw "The spline needs at least 3 points.";
    }

    int N = n_points - 1;


    double h[MAX_NODES];
    for (int i = 0; i < N; i++) {
        h[i] = x[i+1] - x[i];
    }


    double diag[MAX_NODES];
    double lower[MAX_NODES];
    double upper[MAX_NODES];
    double rhs[MAX_NODES];

    for (int i = 0; i < N-2; i++) {
        upper[i] = h[i+1];
        diag[i] = 2.0 * (h[i] + h[i+1]);
        lower[i] = h[i+1];
        rhs[i] = 6.0 * ((y[i+2] - y[i+1])/h[i+1] - (y[i+1] - y[i])/h[i]);
    }


    if (N > 1) {
        diag[N-2] = 2.0 * (h[N-2] + h[N-1]);
        rhs[N-2] = 6.0 * ((y[N] - y[N-1])/h[N-1] - (y[N-1] - y[N-2])/h[N-2]);
    }


    for (int i = 1; i <= N-2; i++) {
        double w = lower[i-1]/diag[i-1];
        diag[i] = diag[i] - w * upper[i-1];
        rhs[i] = rhs[i] - w * rhs[i-1];
    }


    double m[MAX_NODES] = {0};
    if (N > 1) {
        m[N-2] = rhs[N-2]/diag[N-2];
        for (int i = N-3; i >= 0; i--) {
            m[i] = (rhs[i] - upper[i] * m[i+1]) / diag[i];
        }
    }


    double M[MAX_NODES] = {0};
    for (int i = 1; i < N; i++) {
        M[i] = m[i-1];
    }


    if (x_value < x[0]) {
        throw "x is less than the smallest possible x.";
    }
    if (x_value > x[N]) {
        throw "x is bigger than the largest possible x.";
    }


    int j = 0;
    for (int i = 1; i < N+1; i++) {
        if (x_value <= x[i]) {
            j = i - 1;
            break;
        }
    }


    double y_value = M[j] / (6.0 * h[j]) * pow((x[j+1] - x_value), 3.0)
                   + M[j+1] / (6.0 * h[j]) * pow((x_value - x[j]), 3.0)
                   + (y[j] / h[j] - M[j] * h[j] / 6.0) * (x[j+1] - x_value)
                   + (y[j+1] / h[j] - M[j+1] * h[j] / 6.0) * (x_value - x[j]);

    return y_value;
}



int main() {
    try {

        double target_T = 900.0;
        double target_K = 108.0;


        const int MAX_ROWS = 100;
        double T_data[MAX_ROWS];
        double K_data[MAX_ROWS];
        double IV_data[MAX_ROWS];
        int total_rows = 0;


        ifstream file("cleaned_options.csv");
        if (!file.is_open()) {
            throw "Could not open cleaned_options.csv. Make sure it is in the same folder.";
        }

        string line;
        getline(file, line);

        while (getline(file, line) && total_rows < MAX_ROWS) {
            stringstream ss(line);
            string cell;

            getline(ss, cell, ',');
            T_data[total_rows] = stod(cell);

            getline(ss, cell, ',');
            K_data[total_rows] = stod(cell);

            getline(ss, cell, ',');
            IV_data[total_rows] = stod(cell);

            total_rows++;
        }
        file.close();


        double unique_T[20];
        int m = 0;

        for (int i = 0; i < total_rows; i++) {
            bool found = false;
            for (int j = 0; j < m; j++) {
                if (T_data[i] == unique_T[j]) {
                    found = true;
                    break;
                }
            }
            if (!found) {
                unique_T[m] = T_data[i];
                m++;
            }
        }




        double v[20];

        for (int j = 0; j < m; j++) {
            double current_T = unique_T[j];


            double K_temp[20];
            double IV_temp[20];
            int n_j = 0;


            for (int i = 0; i < total_rows; i++) {
                if (T_data[i] == current_T) {
                    K_temp[n_j] = K_data[i];
                    IV_temp[n_j] = IV_data[i];
                    n_j++;
                }
            }


            v[j] = func_cubic_spline(target_K, K_temp, IV_temp, n_j);

            cout << "Maturity T_" << j+1 << " = " << current_T
                 << " days \t-> Interpolated IV v_" << j+1 << " = " << v[j] << "\n";
        }




        double final_iv = func_cubic_spline(target_T, unique_T, v, m);



        ofstream vis_file("visualization_data.csv");
        if (vis_file.is_open()) {
            vis_file << "Maturity,Strike,ImpliedVolatility\n";


            for (int i = 0; i < total_rows; i++) {
                if (T_data[i] == 896.0) {
                    vis_file << 896.0 << "," << K_data[i] << "," << IV_data[i] << "\n";
                }
            }


            for (int i = 0; i < total_rows; i++) {
                if (T_data[i] == 924.0) {
                    vis_file << 924.0 << "," << K_data[i] << "," << IV_data[i] << "\n";
                }
            }


            vis_file << target_T << "," << target_K << "," << final_iv << "\n";

            vis_file.close();
            cout << "created.\n";
        } else {
            cerr << "Could not create .\n";
        }

    } catch (const char* msg) {
        cout << "Error: " << msg << "\n";
    }

    return 0;
}

Overwriting main.cpp


In [10]:
!g++ main.cpp -o run_spline && ./run_spline

Maturity T_1 = 833 days 	-> Interpolated IV v_1 = 0.945599
Maturity T_2 = 861 days 	-> Interpolated IV v_2 = 0.872689
Maturity T_3 = 896 days 	-> Interpolated IV v_3 = 0.837159
Maturity T_4 = 924 days 	-> Interpolated IV v_4 = 0.800587
Maturity T_5 = 952 days 	-> Interpolated IV v_5 = 0.76751
Maturity T_6 = 1015 days 	-> Interpolated IV v_6 = 0.730976
Maturity T_7 = 1105 days 	-> Interpolated IV v_7 = 0.691103
created.
